In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    wheels_dir = f'{temp_dir}/wheels'
    os.makedirs(temp_dir, exist_ok=True)

    if not os.path.exists(input_archive):
        raise FileNotFoundError(f'Archive not found: {input_archive}')

    # Extract when wheels folder is missing or empty.
    if (not os.path.isdir(wheels_dir)) or (not os.listdir(wheels_dir)):
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)

    if (not os.path.isdir(wheels_dir)) or (not os.listdir(wheels_dir)):
        raise RuntimeError(f'Wheels folder missing after extraction: {wheels_dir}')

    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        wheels_dir,
        'unsloth',
        'trl',
        'vllm',
        'openai_harmony'
    ], check=True)


In [ ]:
set_env(
    input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import os
import sys
import json
import math
import time
import queue
import threading
import contextlib
import subprocess
from typing import Optional
from types import SimpleNamespace
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd
import polars as pl
from openai import OpenAI
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server


In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'nvidia/nemotron-3-super'
    model_path = '/kaggle/input/models/neerbasant/nvidia-nemotron-3-super-120b-a12b-nvfp4/transformers/default/1'
    
    kv_cache_dtype = 'fp8'
    dtype = 'auto'

    high_problem_timeout = 600
    base_problem_timeout = 180

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 32768
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 3
    attempts = 6
    workers = 6
    turns = 32
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    top_p = 0.95
    max_completion_tokens = 8192
    max_completion_tokens_fast = 4096
    max_completion_tokens_slow = 16384
    enable_thinking = True
    low_effort_thinking = False
    force_nonempty_content = True

    python_error_limit = 20
    progress_log_every_n_turns = 4
    gpu_log_interval_seconds = 5






In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):
        pass

    def apply_chat_template(self, system_prompt: str, user_prompt: str) -> list[dict]:
        return [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]



In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split('\n')
        if not lines:
            return code

        last_line = lines[-1].strip()
        if 'print' in last_line or 'import' in last_line or not last_line or last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'
        return '\n'.join(lines)

    @property
    def tools(self) -> list[dict]:
        return [{
            'type': 'function',
            'function': {
                'name': 'python',
                'description': self._tool_prompt,
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'code': {
                            'type': 'string',
                            'description': 'Python code to execute in the stateful notebook environment.'
                        }
                    },
                    'required': ['code'],
                    'additionalProperties': False
                }
            }
        }]

    def execute_code(self, code: str) -> str:
        self._ensure_session()
        final_script = self._ensure_last_print(code)

        with self._execution_lock:
            try:
                return self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                return f'[ERROR] {exc}'



In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()

        self._preload_model_weights()
        self.reasoning_parser_plugin_path = self._resolve_reasoning_parser_plugin_path()

        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.progress_log_path = '/kaggle/working/inference_progress.log'
        self.reasoning_log_path = '/kaggle/working/full_reasoning.log'
        self.gpu_log_path = '/kaggle/working/gpu_usage.log'
        self._log_lock = threading.Lock()

        with open(self.progress_log_path, 'w', encoding='utf-8') as log_file:
            log_file.write('AIMO3 inference progress log\n')

        with open(self.reasoning_log_path, 'w', encoding='utf-8') as log_file:
            log_file.write('AIMO3 full reasoning log\n')

        with open(self.gpu_log_path, 'w', encoding='utf-8') as log_file:
            log_file.write('timestamp,total_mb,used_mb,free_mb,gpu_util_pct,mem_util_pct\n')

        self._start_gpu_memory_logger()

    def _resolve_reasoning_parser_plugin_path(self) -> str:
        parser_path = '/kaggle/input/datasets/ramkumar86/super-v3-reasoning-parser-py/super_v3_reasoning_parser.py'

        if not os.path.exists(parser_path):
            raise FileNotFoundError(
                f'Reasoning parser plugin not found at: {parser_path}'
            )

        return parser_path

    def _log_progress(self, message: str) -> None:
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        with self._log_lock:
            with open(self.progress_log_path, 'a', encoding='utf-8') as log_file:
                log_file.write(f'[{timestamp}] {message}\n')

    def _log_reasoning(self, message: str) -> None:
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        with self._log_lock:
            with open(self.reasoning_log_path, 'a', encoding='utf-8') as log_file:
                log_file.write(f'[{timestamp}] {message}\n')

    def _start_gpu_memory_logger(self) -> None:
        def _gpu_logger_loop():
            while True:
                try:
                    output = subprocess.check_output(
                        [
                            'nvidia-smi',
                            '--query-gpu=memory.total,memory.used,memory.free,utilization.gpu,utilization.memory',
                            '--format=csv,noheader,nounits'
                        ],
                        text=True
                    ).strip()
                    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
                    with self._log_lock:
                        with open(self.gpu_log_path, 'a', encoding='utf-8') as log_file:
                            log_file.write(f'{timestamp},{output}\n')
                except Exception:
                    pass

                time.sleep(self.cfg.gpu_log_interval_seconds)

        thread = threading.Thread(target=_gpu_logger_loop, daemon=True)
        thread.start()

    def _preload_model_weights(self) -> None:
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            '--quantization',
            'modelopt',
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--async-scheduling',
            '--enable-prefix-caching',
            '--trust-remote-code',
            '--enable-auto-tool-choice',
            '--tool-call-parser',
            'qwen3_coder',
            '--reasoning-parser-plugin',
            self.reasoning_parser_plugin_path,
            '--reasoning-parser',
            'super_v3'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        if not logprobs_buffer:
            return float('inf')

        total_entropy = 0.0
        token_count = 0

        for top_logprobs in logprobs_buffer:
            if not top_logprobs:
                continue

            token_entropy = 0.0
            for item in top_logprobs:
                log_prob = getattr(item, 'logprob', None)
                if log_prob is None:
                    continue
                prob = math.exp(log_prob)
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)

            if token_entropy > 0:
                total_entropy += token_entropy
                token_count += 1

        if token_count == 0:
            return float('inf')

        return total_entropy / token_count

    def _extract_python_code(self, arguments: str) -> str:
        if not arguments:
            return ''

        try:
            payload = json.loads(arguments)
            if isinstance(payload, dict) and 'code' in payload:
                return str(payload.get('code') or '')
        except Exception:
            pass

        return arguments

    def _append_assistant_message(self, messages: list[dict], message) -> None:
        assistant_msg = {'role': 'assistant', 'content': getattr(message, 'content', '') or ''}
        tool_calls = getattr(message, 'tool_calls', None)
        if tool_calls:
            serialized = []
            for tc in tool_calls:
                if hasattr(tc, 'model_dump'):
                    serialized.append(tc.model_dump())
                elif isinstance(tc, dict):
                    serialized.append(tc)
                else:
                    fn = getattr(tc, 'function', None)
                    serialized.append({
                        'id': getattr(tc, 'id', None),
                        'type': 'function',
                        'function': {
                            'name': getattr(fn, 'name', None),
                            'arguments': getattr(fn, 'arguments', '')
                        }
                    })
            assistant_msg['tool_calls'] = serialized
        messages.append(assistant_msg)

    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float
    ) -> dict:
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Entropy': float('inf')
            }

        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        logprobs_buffer = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
        attempt_max_tokens = (
            self.cfg.max_completion_tokens_fast
            if attempt_index < (self.cfg.attempts // 2)
            else self.cfg.max_completion_tokens_slow
        )
        self._log_progress(
            f'Attempt start | attempt={attempt_index + 1} | seed={attempt_seed} | max_tokens={attempt_max_tokens}'
        )

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox
            )

            messages = self.template.apply_chat_template(system_prompt, problem)

            for turn_idx in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    self._log_progress(f'Attempt stop_event/deadline | attempt={attempt_index + 1} | turn={turn_idx + 1}')
                    break

                if turn_idx == 0 or ((turn_idx + 1) % self.cfg.progress_log_every_n_turns == 0):
                    self._log_progress(f'Turn heartbeat | attempt={attempt_index + 1} | turn={turn_idx + 1} | tokens={total_tokens} | py_calls={python_calls} | py_errors={python_errors}')

                stream = self.client.chat.completions.create(
                    model=self.cfg.served_model_name,
                    messages=messages,
                    tools=local_tool.tools,
                    tool_choice='auto',
                    temperature=self.cfg.temperature,
                    top_p=self.cfg.top_p,
                    max_tokens=attempt_max_tokens,
                    seed=attempt_seed,
                    logprobs=True,
                    top_logprobs=self.cfg.top_logprobs,
                    stream=True,
                    extra_body={
                        'chat_template_kwargs': {
                            'enable_thinking': self.cfg.enable_thinking,
                            'low_effort': self.cfg.low_effort_thinking,
                            'force_nonempty_content': self.cfg.force_nonempty_content,
                        }
                    }
                )

                assistant_parts = []
                reasoning_parts = []
                scan_parts = []
                tool_calls_acc = {}
                latest_usage = None
                tool_call_seen = False

                with stream as response_stream:
                    for chunk in response_stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        usage = getattr(chunk, 'usage', None)
                        if usage is not None:
                            latest_usage = usage

                        if not getattr(chunk, 'choices', None):
                            continue

                        choice = chunk.choices[0]
                        delta = getattr(choice, 'delta', None)
                        if delta is None:
                            continue

                        content_piece = getattr(delta, 'content', '') or ''
                        reasoning_piece = (
                            getattr(delta, 'reasoning_content', '')
                            or getattr(delta, 'reasoning', '')
                            or ''
                        )

                        if content_piece:
                            assistant_parts.append(content_piece)
                            scan_parts.append(content_piece)

                        if reasoning_piece:
                            reasoning_parts.append(reasoning_piece)
                            scan_parts.append(reasoning_piece)

                        delta_tool_calls = getattr(delta, 'tool_calls', None)
                        if delta_tool_calls:
                            tool_call_seen = True
                            for tc in delta_tool_calls:
                                idx = getattr(tc, 'index', 0) or 0
                                entry = tool_calls_acc.setdefault(idx, {
                                    'id': None,
                                    'type': 'function',
                                    'function': {'name': '', 'arguments': ''}
                                })
                                tc_id = getattr(tc, 'id', None)
                                if tc_id:
                                    entry['id'] = tc_id

                                fn = getattr(tc, 'function', None)
                                if fn is not None:
                                    fn_name = getattr(fn, 'name', None)
                                    if fn_name:
                                        entry['function']['name'] = fn_name
                                    fn_args = getattr(fn, 'arguments', None)
                                    if fn_args:
                                        entry['function']['arguments'] += fn_args

                        chunk_logprobs = getattr(choice, 'logprobs', None)
                        if chunk_logprobs is not None:
                            content_parts = getattr(chunk_logprobs, 'content', None)
                            if content_parts:
                                for item in content_parts:
                                    top_logprobs = getattr(item, 'top_logprobs', None)
                                    if top_logprobs:
                                        logprobs_buffer.append(top_logprobs)

                        if final_answer is None and scan_parts:
                            combined = ''.join(scan_parts[-50:])
                            answer = self._scan_for_answer(combined)
                            if answer is not None and not tool_call_seen:
                                final_answer = answer
                                self._log_progress(
                                    f'Answer detected (stream) | attempt={attempt_index + 1} | turn={turn_idx + 1} | answer={final_answer}'
                                )
                                break

                if latest_usage is not None:
                    total_tokens += int(getattr(latest_usage, 'completion_tokens', 0) or 0)

                assistant_text = ''.join(assistant_parts)
                reasoning_content = ''.join(reasoning_parts)

                answer_text_candidates = []
                if reasoning_content:
                    answer_text_candidates.append(reasoning_content)
                if assistant_text:
                    answer_text_candidates.append(assistant_text)

                search_text = '\n'.join(answer_text_candidates)
                if search_text:
                    self._log_reasoning(
                        f"Problem turn | attempt={attempt_index + 1} | turn={turn_idx + 1} | reasoning={None!r} | reasoning_content={reasoning_content!r} | assistant={assistant_text!r}"
                    )
                    if final_answer is None:
                        final_answer = self._scan_for_answer(search_text)

                tool_calls = []
                if tool_calls_acc:
                    for idx in sorted(tool_calls_acc.keys()):
                        data = tool_calls_acc[idx]
                        tool_calls.append(
                            SimpleNamespace(
                                id=data.get('id'),
                                function=SimpleNamespace(
                                    name=data.get('function', {}).get('name', ''),
                                    arguments=data.get('function', {}).get('arguments', '')
                                )
                            )
                        )

                message = SimpleNamespace(
                    content=assistant_text,
                    reasoning=None,
                    reasoning_content=reasoning_content,
                    tool_calls=tool_calls,
                )

                if final_answer is not None and not tool_calls:
                    self._log_progress(f'Answer detected | attempt={attempt_index + 1} | turn={turn_idx + 1} | answer={final_answer}')
                    break

                tool_calls = getattr(message, 'tool_calls', None)
                self._append_assistant_message(messages, message)

                if not tool_calls:
                    break

                for tool_call in tool_calls:
                    if stop_event.is_set() or time.time() > deadline:
                        break

                    if tool_call.function.name != 'python':
                        tool_result = '[ERROR] Unsupported tool call.'
                    else:
                        python_calls += 1
                        code = self._extract_python_code(tool_call.function.arguments)
                        self._log_reasoning(
                            f"Tool call | attempt={attempt_index + 1} | turn={turn_idx + 1} | tool=python | code={code!r}"
                        )
                        tool_result = local_tool.execute_code(code)
                        self._log_reasoning(
                            f"Tool result | attempt={attempt_index + 1} | turn={turn_idx + 1} | tool=python | output={tool_result!r}"
                        )

                        if tool_result.startswith('[ERROR]') or 'Traceback' in tool_result or 'Error:' in tool_result:
                            python_errors += 1

                    messages.append({
                        'role': 'tool',
                        'tool_call_id': tool_call.id,
                        'name': tool_call.function.name,
                        'content': tool_result
                    })

                    if python_errors > self.cfg.python_error_limit:
                        self._log_progress(f'Python error limit reached | attempt={attempt_index + 1} | py_errors={python_errors}')
                        break

                if python_errors > self.cfg.python_error_limit:
                    self._log_progress(f'Attempt aborted by python error limit | attempt={attempt_index + 1} | py_errors={python_errors}')
                    break

        except Exception as exc:
            python_errors += 1
            self._log_progress(f'Attempt exception | attempt={attempt_index + 1} | error={exc}')

        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        mean_entropy = self._compute_mean_entropy(logprobs_buffer)

        self._log_progress(f'Attempt end | attempt={attempt_index + 1} | answer={final_answer} | tokens={total_tokens} | py_calls={python_calls} | py_errors={python_errors} | entropy={mean_entropy:.4f}')

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Answer': final_answer
        }

    def _select_answer(self, detailed_results: list) -> int:
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']

            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:
        print(f'\nProblem: {problem}\n')
        self._log_progress(f'Problem start | text={problem[:200]!r}')

        user_input = f'{problem} {self.cfg.preference_prompt}'

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
        self._log_progress(f'Budget assigned | seconds={budget:.2f} | attempts={self.cfg.attempts} | workers={self.cfg.workers}')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        self._log_progress(f'Early stop triggered | votes={counts[0][1]} | answer={counts[0][0]}')
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            display(results_dataframe)

        if not valid_answers:
            self._log_progress('Problem end | no valid answers | result=0')
            print('\nResult: 0\n')
            return 0

        selected = self._select_answer(detailed_results)
        self._log_progress(f'Problem end | selected_answer={selected} | valid_answers={len(valid_answers)}')
        return selected

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass














In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )